# 04 - Embeddings e indexación vectorial con BGE-M3

Este notebook genera embeddings a partir de los chunks estructurados del corpus normativo ambiental e indexa los resultados en **Qdrant**.

Esta versión está adaptada para usar principalmente:

```text
BAAI/bge-m3 + FlagEmbedding
```

La decisión se debe a que BGE-M3 es un modelo multilingüe y adecuado para recuperación semántica. Sin embargo, como puede ser pesado para algunas laptops, el notebook está preparado con configuración conservadora:

- `BATCH_SIZE = 1`
- `MAX_LENGTH = 512`
- uso de CPU por defecto si no hay GPU
- `fp16` solo si hay CUDA
- normalización manual de vectores
- alternativa configurable con `sentence-transformers`

Entrada principal:

- `data/chunks/chunks_normativa_v1.jsonl`

Salida generada:

- colección vectorial en Qdrant;
- reporte de indexación en `experiments/resultados/`;
- prueba inicial de búsqueda semántica top-k.

Este paso permite pasar de documentos segmentados a una base consultable por similitud semántica.


## 0. Requisitos previos

### 0.1. Instalar dependencias

Ejecutar en terminal, dentro del entorno virtual:

```bash
pip install -U pandas numpy tqdm qdrant-client FlagEmbedding torch
```

Si se usará la alternativa ligera con `sentence-transformers`:

```bash
pip install -U sentence-transformers
```

### 0.2. Levantar Qdrant

Antes de indexar, Qdrant debe estar corriendo.

Con Docker:

```bash
docker run -p 6333:6333 -p 6334:6334 qdrant/qdrant
```

Luego puedes revisar el panel en:

```text
http://localhost:6333/dashboard
```


## 1. Importación de librerías

In [4]:
import os
from google.colab import drive

# Ensure the mount point is empty before mounting
if os.path.exists('/content/drive'):
    try:
        # Attempt to unmount if already mounted
        drive.flush_and_unmount()
    except ValueError:
        pass # Drive might not be mounted, proceed to remove directory
    # Remove the directory and its contents if it exists and is not empty
    # This is a robust way to ensure a clean mount point
    !rm -rf /content/drive/*
    !rmdir /content/drive || true # Remove directory, ignore error if not empty after rm -rf

# Create the mount point directory if it doesn't exist (rmdir might remove it)
os.makedirs('/content/drive', exist_ok=True)

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
# ============================================================
# 0. Instalación de dependencias
# ============================================================
!pip install -q --upgrade pip
!pip uninstall -y transformers FlagEmbedding torchvision torchaudio
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.44.2
!pip install -q FlagEmbedding==1.2.11
!pip install -q sentence-transformers==3.0.1
!pip install -q qdrant-client pandas numpy tqdm

print("✅ Instalación completa — reinicia la sesión antes de continuar")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.1 MB/s eta 0:00:00
Found existing installation: transformers 5.9.0
Uninstalling transformers-5.9.0:
  Successfully uninstalled transformers-5.9.0
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.19.1 requires transformers, which is not installed.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, which is not installed.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Instalación completa — reinicia la sesión antes de continuar

In [6]:
# ============================================================
# 1. Importación de librerías
# ============================================================

from pathlib import Path
from datetime import datetime

import json
import time
import gc
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    import pandas as pd
    print("pandas importado correctamente.")
except ImportError:
    raise ImportError("Instalar: pip install pandas")

try:
    import numpy as np
    print("numpy importado correctamente.")
except ImportError:
    raise ImportError("Instalar: pip install numpy")

try:
    from tqdm.auto import tqdm
    print("tqdm importado correctamente.")
except ImportError:
    raise ImportError("Instalar: pip install tqdm")

try:
    import torch
    print("torch importado correctamente.")
except ImportError:
    raise ImportError("Instalar: pip install torch")

try:
    from qdrant_client import QdrantClient
    from qdrant_client.models import (
        Distance,
        VectorParams,
        PointStruct
    )
    print("qdrant-client importado correctamente.")
except ImportError:
    raise ImportError("Instalar: pip install qdrant-client")

print("Librerías cargadas correctamente.")


pandas importado correctamente.
numpy importado correctamente.
tqdm importado correctamente.
torch importado correctamente.
qdrant-client importado correctamente.
Librerías cargadas correctamente.


## 2. Definición de rutas del proyecto

El notebook puede ejecutarse desde la raíz del repositorio o desde la carpeta `notebooks/`.


In [7]:
from pathlib import Path

# Explicitly set ROOT_DIR to the Google Drive path
ROOT_DIR = Path("/content/drive/MyDrive/RAG_Normativa_Ambiental")

DATA_DIR = ROOT_DIR / "data"
CHUNKS_DIR = DATA_DIR / "chunks"
REPORTS_DIR = ROOT_DIR / "experiments" / "resultados"

CHUNKS_JSONL_PATH = CHUNKS_DIR / "chunks_normativa_v1.jsonl"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"CHUNKS_JSONL_PATH: {CHUNKS_JSONL_PATH}")
print(f"REPORTS_DIR: {REPORTS_DIR}")

ROOT_DIR: /content/drive/MyDrive/RAG_Normativa_Ambiental
CHUNKS_JSONL_PATH: /content/drive/MyDrive/RAG_Normativa_Ambiental/data/chunks/chunks_normativa_v1.jsonl
REPORTS_DIR: /content/drive/MyDrive/RAG_Normativa_Ambiental/experiments/resultados


In [8]:
print(f"Verificando la existencia del archivo: {CHUNKS_JSONL_PATH}")
!ls -l {CHUNKS_JSONL_PATH.parent}
!ls -l {CHUNKS_JSONL_PATH}

Verificando la existencia del archivo: /content/drive/MyDrive/RAG_Normativa_Ambiental/data/chunks/chunks_normativa_v1.jsonl
total 4573
-rw------- 1 root root 2087708 May 20 02:34 chunks_normativa_v1.csv
-rw------- 1 root root 2594455 May 20 02:34 chunks_normativa_v1.jsonl
-rw------- 1 root root 2594455 May 20 02:34 /content/drive/MyDrive/RAG_Normativa_Ambiental/data/chunks/chunks_normativa_v1.jsonl


In [9]:
# Check Google Drive mount status
!df -h /content/drive

# List contents of the root Google Drive path
print(f"Listing contents of ROOT_DIR: {ROOT_DIR}")
!ls -l {ROOT_DIR}

# List contents of the data directory
print(f"Listing contents of DATA_DIR: {DATA_DIR}")
!ls -l {DATA_DIR}

# List contents of the chunks directory
print(f"Listing contents of CHUNKS_DIR: {CHUNKS_DIR}")
!ls -l {CHUNKS_DIR}

Filesystem      Size  Used Avail Use% Mounted on
drive           108G   32G   76G  30% /content/drive
Listing contents of ROOT_DIR: /content/drive/MyDrive/RAG_Normativa_Ambiental
total 20
drwx------ 3 root root 4096 Jun  8 21:44 data
drwx------ 3 root root 4096 Jun  8 22:06 experiments
drwx------ 2 root root 4096 Jun  9 02:35 notebooks
drwx------ 2 root root 4096 Jun  9 03:10 Notebooks
drwx------ 2 root root 4096 Jun  9 02:44 qdrant_storage
Listing contents of DATA_DIR: /content/drive/MyDrive/RAG_Normativa_Ambiental/data
total 20
drwx------ 2 root root 4096 Jun  8 21:44 chunks
drwx------ 2 root root 4096 Jun  8 21:44 embeddings
drwx------ 2 root root 4096 Jun  8 21:44 metadata
drwx------ 2 root root 4096 Jun  8 21:44 processed
drwx------ 2 root root 4096 Jun  8 21:44 raw
Listing contents of CHUNKS_DIR: /content/drive/MyDrive/RAG_Normativa_Ambiental/data/chunks
total 4573
-rw------- 1 root root 2087708 May 20 02:34 chunks_normativa_v1.csv
-rw------- 1 root root 2594455 May 20 02:34 chun

## 3. Configuración general del modelo y Qdrant

Por defecto se usa:

```text
BAAI/bge-m3 con FlagEmbedding
```

Si tu máquina no soporta BGE-M3, puedes cambiar el backend a:

```python
EMBEDDING_BACKEND = "sentence_transformers"
MODEL_NAME = "intfloat/multilingual-e5-small"
```

Pero para el proyecto final se recomienda intentar mantener BGE-M3, al menos en Colab o en una máquina con más recursos.


In [10]:
# ============================================================
# 3. Configuración general
# ============================================================

# Opciones:
# - "bge_m3_flagembedding"  -> recomendado para BAAI/bge-m3
# - "sentence_transformers" -> alternativa ligera
EMBEDDING_BACKEND = "bge_m3_flagembedding"

# Modelo principal del proyecto
MODEL_NAME = "BAAI/bge-m3"

# Alternativa ligera si BGE-M3 no corre localmente:
# EMBEDDING_BACKEND = "sentence_transformers"
# MODEL_NAME = "intfloat/multilingual-e5-small"

COLLECTION_NAME = "normativa_ambiental_chunks_v1"

QDRANT_MODE = "memory"  # opciones: "memory" o "docker"

QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

# Para BGE-M3 en laptop local, usar batch pequeño
BATCH_SIZE = 1
TOP_K = 5

# Longitud máxima del texto que entra al modelo
# Si tu máquina se queda corta, puedes bajar a 384.
MAX_LENGTH = 512

# Prefijos:
# BGE-M3 no requiere prefijos obligatorios.
# E5 sí suele beneficiarse de "passage:" y "query:".
if EMBEDDING_BACKEND == "sentence_transformers" and "e5" in MODEL_NAME.lower():
    PASSAGE_PREFIX = "passage: "
    QUERY_PREFIX = "query: "
else:
    PASSAGE_PREFIX = ""
    QUERY_PREFIX = ""

print(f"Backend de embeddings: {EMBEDDING_BACKEND}")
print(f"Modelo de embeddings: {MODEL_NAME}")
print(f"Colección Qdrant: {COLLECTION_NAME}")
print(f"Qdrant: {QDRANT_HOST}:{QDRANT_PORT}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max length: {MAX_LENGTH}")


Backend de embeddings: bge_m3_flagembedding
Modelo de embeddings: BAAI/bge-m3
Colección Qdrant: normativa_ambiental_chunks_v1
Qdrant: localhost:6333
Batch size: 1
Max length: 512


## 4. Carga de chunks generados

In [11]:
if not CHUNKS_JSONL_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo de chunks: {CHUNKS_JSONL_PATH}. "
        "Ejecuta primero el Notebook 03 de chunking estructural."
    )

chunks = []

with open(CHUNKS_JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))

print(f"Total de chunks cargados: {len(chunks)}")

chunks_df = pd.DataFrame(chunks)
chunks_df.head()


Total de chunks cargados: 1057


,id_chunk,id_documento,archivo_pdf,archivo_txt,titulo_documento,tipo_norma,numero_norma,entidad_emisora,fecha_publicacion,tema_principal,...,titulo_seccion,capitulo,anexo,pagina_inicio,pagina_fin,orden_chunk,suborden,n_palabras,n_caracteres,texto
0,DOC-001_CHK-0001,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,1,1,453,2860,"NORMAS LEGALES El Peruano Lima, jueves 6 de se..."
1,DOC-001_CHK-0001_P02,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,1,2,295,1878,así como en el proceso al que se reﬁ ere el De...
2,DOC-001_CHK-0002,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,2,1,58,402,Artículo 1º.- Del objeto de la norma\nApruébes...
3,DOC-001_CHK-0003,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,3,1,25,144,Artículo 2º.- De la vigencia\nEl presente Decr...
4,DOC-001_CHK-0004,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,4,1,51,311,Artículo 3º.- Del refrendo\nEl presente Decret...


## 5. Validación básica de chunks

In [12]:
required_chunk_fields = [
    "id_chunk",
    "id_documento",
    "titulo_documento",
    "tipo_norma",
    "numero_norma",
    "tema_principal",
    "subtema",
    "texto",
]

missing_fields = [field for field in required_chunk_fields if field not in chunks_df.columns]

if missing_fields:
    raise ValueError(f"Faltan campos obligatorios en los chunks: {missing_fields}")

empty_text_chunks = chunks_df[
    chunks_df["texto"].isna() |
    (chunks_df["texto"].astype(str).str.strip() == "")
]

if not empty_text_chunks.empty:
    print(f"Advertencia: existen {len(empty_text_chunks)} chunks sin texto. Serán excluidos.")
    chunks_df = chunks_df[~chunks_df.index.isin(empty_text_chunks.index)].copy()

chunks_df = chunks_df.reset_index(drop=True)

print(f"Chunks válidos para embeddings: {len(chunks_df)}")

if "n_palabras" in chunks_df.columns:
    print("Distribución de palabras por chunk:")
    display(chunks_df["n_palabras"].describe())


Chunks válidos para embeddings: 1057
Distribución de palabras por chunk:


,n_palabras
count,1057.000000
mean,210.133396
std,163.340571
min,25.000000
25%,72.000000
50%,139.000000
75%,400.000000
max,635.000000


## 6. Carga del modelo de embeddings

Esta versión usa BGE-M3 mediante `FlagEmbedding`.

Si el kernel se vuelve a caer al cargar BGE-M3, las opciones recomendadas son:

1. Ejecutar este notebook en Google Colab con GPU.
2. Reducir `MAX_LENGTH` de 512 a 384.
3. Usar temporalmente `intfloat/multilingual-e5-small` para desarrollo local.


In [13]:
# ============================================================
# 6. Carga del modelo de embeddings
# ============================================================

def load_embedding_model(backend: str, model_name: str):
    start_time = time.time()

    if backend == "bge_m3_flagembedding":
        try:
            from FlagEmbedding import BGEM3FlagModel
        except ImportError:
            raise ImportError(
                "No se encontró FlagEmbedding. Instala con: pip install -U FlagEmbedding"
            )

        device = "cuda" if torch.cuda.is_available() else "cpu"
        use_fp16 = True if device == "cuda" else False

        print(f"Dispositivo usado: {device}")
        print(f"FP16 activado: {use_fp16}")

        # Algunas versiones de FlagEmbedding aceptan device, otras no.
        try:
            model = BGEM3FlagModel(
                model_name,
                use_fp16=use_fp16,
                device=device
            )
        except TypeError:
            print("La versión instalada de FlagEmbedding no acepta el parámetro device. Se cargará sin ese parámetro.")
            model = BGEM3FlagModel(
                model_name,
                use_fp16=use_fp16
            )

        load_time = time.time() - start_time
        return model, load_time

    elif backend == "sentence_transformers":
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            raise ImportError(
                "No se encontró sentence-transformers. Instala con: pip install sentence-transformers"
            )

        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Dispositivo usado: {device}")

        model = SentenceTransformer(model_name, device=device)
        load_time = time.time() - start_time
        return model, load_time

    else:
        raise ValueError(f"Backend no reconocido: {backend}")


model, load_time = load_embedding_model(EMBEDDING_BACKEND, MODEL_NAME)

print(f"Modelo cargado correctamente en {load_time:.2f} segundos.")


Dispositivo usado: cpu
FP16 activado: False


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

Modelo cargado correctamente en 79.08 segundos.


/usr/local/lib/python3.12/dist-packages/FlagEmbedding/BGE_M3/modeling.py:335: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  colbert_state_dict = torch.load(os.path.join(mode

## 7. Funciones de generación y normalización de embeddings

In [14]:
def l2_normalize(vectors: np.ndarray) -> np.ndarray:
    vectors = np.asarray(vectors, dtype=np.float32)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return vectors / norms


def encode_texts(texts, is_query: bool = False):
    '''
    Genera embeddings usando el backend configurado.
    Devuelve un numpy array normalizado.
    '''
    prefix = QUERY_PREFIX if is_query else PASSAGE_PREFIX
    prepared_texts = [prefix + str(text) for text in texts]

    if EMBEDDING_BACKEND == "bge_m3_flagembedding":
        output = model.encode(
            prepared_texts,
            batch_size=BATCH_SIZE,
            max_length=MAX_LENGTH,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False
        )

        dense_vecs = output["dense_vecs"]
        return l2_normalize(np.array(dense_vecs, dtype=np.float32))

    elif EMBEDDING_BACKEND == "sentence_transformers":
        embeddings = model.encode(
            prepared_texts,
            normalize_embeddings=True,
            batch_size=BATCH_SIZE,
            convert_to_numpy=True,
            show_progress_bar=False
        )
        return np.array(embeddings, dtype=np.float32)

    else:
        raise ValueError(f"Backend no reconocido: {EMBEDDING_BACKEND}")


# Prueba para obtener dimensión del vector
test_embedding = encode_texts(["Texto de prueba"], is_query=False)
VECTOR_SIZE = len(test_embedding[0])

print(f"Dimensión del vector: {VECTOR_SIZE}")


Dimensión del vector: 1024


## 8. Preparación de textos para embeddings

Se combina el texto del chunk con metadatos útiles para reforzar el contexto semántico.

Esto ayuda a que el embedding no represente solo el contenido aislado, sino también información normativa como título, tipo de norma, número, tema y sección.


In [15]:
def build_embedding_text(row) -> str:
    metadata_context = (
        f"Título del documento: {row.get('titulo_documento', 'No determinado')}\n"
        f"Tipo de norma: {row.get('tipo_norma', 'No determinado')}\n"
        f"Número de norma: {row.get('numero_norma', 'No determinado')}\n"
        f"Entidad emisora: {row.get('entidad_emisora', 'No determinado')}\n"
        f"Tema: {row.get('tema_principal', 'No determinado')}\n"
        f"Subtema: {row.get('subtema', 'No determinado')}\n"
        f"Sección: {row.get('seccion', 'No determinado')}\n"
    )
    return metadata_context + "\nContenido:\n" + str(row.get("texto", ""))


chunks_df["texto_embedding"] = chunks_df.apply(build_embedding_text, axis=1)

print("Ejemplo de texto usado para embedding:")
print(chunks_df.loc[0, "texto_embedding"][:1500])


Ejemplo de texto usado para embedding:
Título del documento: Aprueban Disposiciones Complementarias para el Instrumento de Gestión Ambiental Correctivo - IGAC para la Formalización de Actividades de Pequeña Minería y Minería Artesanal en curso
Tipo de norma: Decreto Supremo
Número de norma: Decreto Supremo N.º 004-2012-MINAM
Entidad emisora: MINAM
Tema: Gestión ambiental
Subtema: Minería y ambiente / IGAC
Sección: Preámbulo / considerandos

Contenido:
NORMAS LEGALES El Peruano Lima, jueves 6 de setiembre de 2012 474028 Aprueban Disposiciones Complemen- tarias para el Instrumento de Gestión Ambiental Correctivo (IGAC), para la Formalización de Actividades de Pe- queña Minería y Minería Artesanal en curso DECRETO SUPREMO Nº 004-2012-MINAM EL PRESIDENTE DE LA REPÚBLICA CONSIDERANDO: Que, el derecho fundamental a gozar de un ambiente equilibrado y adecuado al desarrollo de la vida, como lo señala el inciso 22 del artículo 2º de la Constitución Política del Perú, tiene relevancia para las g

## 9. Generación de embeddings

Se generan embeddings por lotes.

Con BGE-M3 en laptop local se recomienda mantener `BATCH_SIZE = 1`.


In [20]:
import numpy as np
from pathlib import Path

EMBEDDINGS_PATH = Path("/content/drive/MyDrive/RAG_Normativa_Ambiental/data/embeddings/bge_m3_embeddings.npy")

if EMBEDDINGS_PATH.exists():
    embeddings = np.load(str(EMBEDDINGS_PATH))
    print(f"✅ Embeddings cargados desde Drive.")
    print(f"   Shape: {embeddings.shape}")
    print(f"   Total chunks: {len(embeddings)}")
else:
    print("❌ No se encontró el archivo. Verificar ruta.")

✅ Embeddings cargados desde Drive.
   Shape: (1057, 1024)
   Total chunks: 1057


In [19]:
texts = chunks_df["texto_embedding"].tolist()

embeddings_list = []

start_time = time.time()

for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="Generando embeddings"):
    batch_texts = texts[start:start + BATCH_SIZE]

    batch_embeddings = encode_texts(batch_texts, is_query=False)
    embeddings_list.extend(batch_embeddings)

    # Limpieza conservadora de memoria
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

embedding_time = time.time() - start_time

embeddings = np.array(embeddings_list, dtype=np.float32)

print(f"Embeddings generados: {embeddings.shape}")
print(f"Tiempo total de generación: {embedding_time:.2f} segundos")
print(f"Tiempo promedio por chunk: {embedding_time / max(len(texts), 1):.4f} segundos")


Generando embeddings:   0%|          | 0/1057 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 10. Conexión con Qdrant

Se verifica que Qdrant esté corriendo en `localhost:6333`.


In [ ]:
from qdrant_client import QdrantClient

from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

In [ ]:
print(PointStruct)

<class 'qdrant_client.http.models.models.PointStruct'>


In [ ]:
from tqdm.auto import tqdm

import pandas as pd
import numpy as np
import torch

from qdrant_client import QdrantClient

from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

In [16]:
# ============================================================
# 9. Configuración y conexión a Qdrant
# ============================================================

from qdrant_client import QdrantClient
from pathlib import Path

# Persistencia en Google Drive
QDRANT_STORAGE = Path("/content/drive/MyDrive/RAG_Normativa_Ambiental/qdrant_storage")
QDRANT_STORAGE.mkdir(parents=True, exist_ok=True)

client = QdrantClient(path=str(QDRANT_STORAGE))
print(f"Qdrant iniciado con persistencia en Drive.")
print(f"Colecciones existentes: {[c.name for c in client.get_collections().collections]}")

Qdrant iniciado con persistencia en Drive.
Colecciones existentes: ['normativa_ambiental_chunks_v1']


## 11. Creación o recreación de la colección

Para evitar duplicados durante pruebas, se recrea la colección.

Si quieres conservar una colección previa, cambia:

```python
RECREATE_COLLECTION = False
```


In [17]:
RECREATE_COLLECTION = True

existing_collections = [collection.name for collection in client.get_collections().collections]

if COLLECTION_NAME in existing_collections and RECREATE_COLLECTION:
    client.delete_collection(collection_name=COLLECTION_NAME)
    print(f"Colección eliminada: {COLLECTION_NAME}")

existing_collections = [collection.name for collection in client.get_collections().collections]

if COLLECTION_NAME not in existing_collections:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,
            distance=Distance.COSINE
        )
    )
    print(f"Colección creada: {COLLECTION_NAME}")
else:
    print(f"Colección ya existente: {COLLECTION_NAME}")


Colección eliminada: normativa_ambiental_chunks_v1
Colección creada: normativa_ambiental_chunks_v1


### 12. Preparación de puntos para Qdrant

Cada chunk se almacena como un punto vectorial.

El vector representa el contenido semántico y el payload conserva los metadatos.


In [21]:
def clean_payload_value(value):
    if pd.isna(value):
        return "No determinado"
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    return value


points = []

for idx, row in chunks_df.iterrows():
    payload = {
        "id_chunk": clean_payload_value(row.get("id_chunk")),
        "id_documento": clean_payload_value(row.get("id_documento")),
        "archivo_pdf": clean_payload_value(row.get("archivo_pdf")),
        "titulo_documento": clean_payload_value(row.get("titulo_documento")),
        "tipo_norma": clean_payload_value(row.get("tipo_norma")),
        "numero_norma": clean_payload_value(row.get("numero_norma")),
        "entidad_emisora": clean_payload_value(row.get("entidad_emisora")),
        "fecha_publicacion": clean_payload_value(row.get("fecha_publicacion")),
        "tema_principal": clean_payload_value(row.get("tema_principal")),
        "subtema": clean_payload_value(row.get("subtema")),
        "fuente_oficial": clean_payload_value(row.get("fuente_oficial")),
        "url_fuente": clean_payload_value(row.get("url_fuente")),
        "estado_vigencia": clean_payload_value(row.get("estado_vigencia")),
        "prioridad": clean_payload_value(row.get("prioridad")),
        "tipo_chunk": clean_payload_value(row.get("tipo_chunk")),
        "seccion": clean_payload_value(row.get("seccion")),
        "titulo_seccion": clean_payload_value(row.get("titulo_seccion")),
        "capitulo": clean_payload_value(row.get("capitulo")),
        "anexo": clean_payload_value(row.get("anexo")),
        "pagina_inicio": clean_payload_value(row.get("pagina_inicio")),
        "pagina_fin": clean_payload_value(row.get("pagina_fin")),
        "n_palabras": clean_payload_value(row.get("n_palabras")),
        "n_caracteres": clean_payload_value(row.get("n_caracteres")),
        "texto": clean_payload_value(row.get("texto")),
    }

    point = PointStruct(
        id=int(idx),
        vector=embeddings[idx].tolist(),
        payload=payload
    )
    points.append(point)

print(f"Puntos preparados para Qdrant: {len(points)}")


Puntos preparados para Qdrant: 1057


## 13. Indexación en Qdrant

Se suben los puntos vectoriales por lotes.

Para BGE-M3 local, se mantiene el batch pequeño.


In [22]:
start_time = time.time()

for start in tqdm(range(0, len(points), BATCH_SIZE), desc="Indexando en Qdrant"):
    batch_points = points[start:start + BATCH_SIZE]
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch_points
    )

indexing_time = time.time() - start_time

print(f"Indexación completada en {indexing_time:.2f} segundos.")
print(f"Total de puntos indexados: {len(points)}")


Indexando en Qdrant:   0%|          | 0/1057 [00:00<?, ?it/s]

Indexación completada en 34.13 segundos.
Total de puntos indexados: 1057


## 14. Verificación de la colección indexada

In [23]:
collection_info = client.get_collection(collection_name=COLLECTION_NAME)
print(collection_info)


status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=1057 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1, prevent_unoptimized=None), wal_config=

## 15. Función de búsqueda semántica

Esta función convierte una consulta en embedding y recupera los chunks más similares desde Qdrant.


In [24]:
def search_semantic(query: str, top_k: int = TOP_K):
    query_vector = encode_texts([query], is_query=True)[0]

    results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector.tolist(),
        limit=top_k,
        with_payload=True
    )

    output = []

    for rank, result in enumerate(results, start=1):
        payload = result.payload

        output.append({
            "rank": rank,
            "score": result.score,
            "id_chunk": payload.get("id_chunk"),
            "id_documento": payload.get("id_documento"),
            "titulo_documento": payload.get("titulo_documento"),
            "tipo_norma": payload.get("tipo_norma"),
            "numero_norma": payload.get("numero_norma"),
            "tema_principal": payload.get("tema_principal"),
            "subtema": payload.get("subtema"),
            "seccion": payload.get("seccion"),
            "pagina_inicio": payload.get("pagina_inicio"),
            "pagina_fin": payload.get("pagina_fin"),
            "texto": payload.get("texto"),
        })

    return output


## 16. Prueba inicial de búsqueda vectorial

In [25]:
# ============================================================
# 14. Búsqueda semántica
# ============================================================

def search_semantic(query, top_k=5):

    # Generar embedding de la consulta
    query_vector = encode_texts(
        [query],
        is_query=True
    )[0]

    # Búsqueda en Qdrant
    search_result = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector.tolist(),
        limit=top_k
    )

    results = []

    for rank, point in enumerate(search_result.points, start=1):

        payload = point.payload

        results.append({
            "rank": rank,
            "score": float(point.score),

            "id_documento":
                payload.get("id_documento", ""),

            "titulo_documento":
                payload.get("titulo_documento", ""),

            "numero_norma":
                payload.get("numero_norma", ""),

            "seccion":
                payload.get("seccion", ""),

            "pagina_inicio":
                payload.get("pagina_inicio", ""),

            "pagina_fin":
                payload.get("pagina_fin", ""),

            "texto":
                payload.get("texto", "")
        })

    return results

In [26]:
test_queries = [
    "¿Qué regula el Instrumento de Gestión Ambiental Correctivo IGAC?",
    "¿Qué normas tratan sobre estándares de calidad ambiental?",
    "¿Qué obligaciones ambientales existen para la pequeña minería?"
]

for query in test_queries:

    print("=" * 120)
    print(f"CONSULTA: {query}")
    print("=" * 120)

    results = search_semantic(
        query=query,
        top_k=5
    )

    for item in results:

        print(
            f"Rank: {item['rank']} | "
            f"Score: {item['score']:.4f}"
        )

        print(
            f"Documento: "
            f"{item['id_documento']} - "
            f"{item['titulo_documento']}"
        )

        print(
            f"Norma: {item['numero_norma']}"
        )

        print(
            f"Sección: {item['seccion']}"
        )

        print(
            f"Páginas: "
            f"{item['pagina_inicio']} - "
            f"{item['pagina_fin']}"
        )

        print("-" * 80)

        print(
            str(item["texto"])[:700]
        )

        print()

CONSULTA: ¿Qué regula el Instrumento de Gestión Ambiental Correctivo IGAC?
Rank: 1 | Score: 0.7322
Documento: DOC-001 - Aprueban Disposiciones Complementarias para el Instrumento de Gestión Ambiental Correctivo - IGAC para la Formalización de Actividades de Pequeña Minería y Minería Artesanal en curso
Norma: Decreto Supremo N.º 004-2012-MINAM
Sección: Artículo 1º.- Objeto
Páginas: 1 - 1
--------------------------------------------------------------------------------
Artículo 1º.- Objeto
El presente dispositivo tiene como objeto regular el
Instrumento de Gestión Ambiental Correctivo – IGAC,
aplicable en los procesos de formalización de las actividades
en curso de pequeña minería y minería artesanal, de
acuerdo a lo establecido en el artículo 9º del Decreto
Legislativo Nº 1105.

Rank: 2 | Score: 0.7251
Documento: DOC-001 - Aprueban Disposiciones Complementarias para el Instrumento de Gestión Ambiental Correctivo - IGAC para la Formalización de Actividades de Pequeña Minería y Minería Art

## 17. Guardado de reporte de indexación

Se guarda un resumen técnico de la indexación realizada.


In [27]:
from datetime import datetime

In [28]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

report = {
    "fecha_indexacion": datetime.now().isoformat(timespec="seconds"),
    "backend_embeddings": EMBEDDING_BACKEND,
    "modelo_embeddings": MODEL_NAME,
    "coleccion_qdrant": COLLECTION_NAME,
    "vector_size": int(VECTOR_SIZE),
    "total_chunks_indexados": int(len(points)),
    "batch_size": int(BATCH_SIZE),
    "max_length": int(MAX_LENGTH),
    "tiempo_carga_modelo_segundos": round(load_time, 4),
    "tiempo_generacion_embeddings_segundos": round(embedding_time, 4),
    "tiempo_indexacion_segundos": round(indexing_time, 4),
    "distancia": "COSINE",
}

report_path = REPORTS_DIR / f"reporte_indexacion_vectorial_{timestamp}.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=4)

print(f"Reporte guardado en: {report_path}")
print(json.dumps(report, ensure_ascii=False, indent=4))


NameError: name 'embedding_time' is not defined

## 18. Resultado final

Si las búsquedas devuelven fragmentos razonables, la indexación vectorial está funcionando correctamente.

Siguiente etapa:

- implementar búsqueda lexical con BM25;
- fusionar resultados con RRF;
- construir búsqueda híbrida;
- integrar el modelo generativo.


In [29]:
print("Notebook 04 completado.")
print("La base vectorial está lista para pruebas de recuperación semántica.")
print("Siguiente paso recomendado: Notebook 05 - Búsqueda híbrida y pruebas RAG.")


Notebook 04 completado.
La base vectorial está lista para pruebas de recuperación semántica.
Siguiente paso recomendado: Notebook 05 - Búsqueda híbrida y pruebas RAG.


In [30]:
info = client.get_collection("normativa_ambiental_chunks_v1")
print(f"Total vectores indexados: {info.points_count}")

Total vectores indexados: 1057
